# Raw dataset cleanup (pre-ODE model fitting)

In [ ]:
# Imports and plotting defaults
import os
from pathlib import Path
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import regex as re
import scienceplots
import numba as nb

plt.style.use("science")

def set_size(width, fraction=1, subplots=(1, 1)):
    """Set figure dimensions to avoid scaling in LaTeX."""
    width_pt_map = {"thesis": 426.79135, "beamer": 307.28987}
    width_pt = width_pt_map.get(width, width)

    fig_width_pt = width_pt * fraction
    inches_per_pt = 1 / 72.27
    golden_ratio = (5 ** 0.5 - 1) / 2

    fig_width_in = fig_width_pt * inches_per_pt
    fig_height_in = fig_width_in * golden_ratio * (subplots[0] / subplots[1])
    return (fig_width_in, fig_height_in)

plt.rcParams["figure.figsize"] = set_size("thesis", fraction=1.15)
plt.rcParams["figure.dpi"] = 300


In [ ]:
# Project modules
import analysis as flua


In [ ]:
# Locate working directory and ensure output folder exists
def detect_working_path():
    candidates = [
        Path(os.getenv("FLUOPTI_WORKDIR", "")),
        Path.cwd(),
        Path.cwd() / "fluopti_fits",
        Path.cwd().parent / "fluopti_fits",
    ]
    for candidate in candidates:
        if (candidate / "all_data").exists() and (candidate / "data_analysis").exists():
            return candidate.resolve()
    raise FileNotFoundError(
        "Could not find a directory containing 'all_data' and 'data_analysis'. "
        "Set FLUOPTI_WORKDIR to the base 'fluopti_fits' path."
    )

working_path = detect_working_path()
os.chdir(working_path)
print(f"Current directory: {working_path}")
(working_path / "data_analysis").mkdir(exist_ok=True)


## Search for pickle files


In [ ]:
TL_DATA_REGEX = r"TL\d{6}"
PICKLED_DATA_DIR = working_path / "all_data"

pickled_files = sorted(PICKLED_DATA_DIR.glob("*.pickle"))
if not pickled_files:
    raise FileNotFoundError(
        f"No pickled data files found in {PICKLED_DATA_DIR} matching the regex '{TL_DATA_REGEX}'."
    )

TRUE_TIMELAPSE_NAMES = [path.stem.rsplit("_", 1)[-1] for path in pickled_files]

# The data added to the timelapse information dictionary below must match the pickled files, and corresponds to the data
# that will be used for estimating the ODE model parameters by bayesian inference.
TIMELAPSE_INFORMATION = {
    "TL240521": ["48HRS_GREEN_20PER", [0.20, 0.00]],
    "TL240523": ["48HRS_GREEN_40PER", [0.40, 0.00]],
    "TL240528": ["48HRS_GREEN_80PER", [0.80, 0.00]],
    "TL240530": ["48HRS_DARK", [0.00, 0.00]],
    "TL240607": ["48HRS_GREEN_RED_100PER", [1.00, 1.00]],
    "TL240610": ["48HRS_RED_50PER", [0.00, 0.50]],
    "TL240615": ["48HRS_RED_75PER", [0.00, 0.75]],
    "TL240716": ["48HRS_RED_10PER_GREEN_100PER", [1.00, 0.10]],
    "TL240719": ["48HRS_RED_10PER_GREEN_50PER", [0.50, 0.10]],
    "TL240727": ["48HRS_RED_10PER_GREEN_25PER", [0.25, 0.10]],
    "TL240801": ["48HRS_RED_20PER_GREEN_50PER", [0.50, 0.20]],
    "TL240804": ["48HRS_RED_30PER_GREEN_50PER", [0.50, 0.30]],
}

missing_pickles = [tl for tl in TIMELAPSE_INFORMATION if tl not in TRUE_TIMELAPSE_NAMES]
if missing_pickles:
    raise ValueError(f"Timelapse IDs missing in pickled files: {', '.join(missing_pickles)}")


## Search for processed .CSV files


In [ ]:
PROCESSED_DATASET_DIR = working_path / "data_analysis"

PROCESSED_DATASETS = sorted(
    path for path in PROCESSED_DATASET_DIR.glob("*.csv")
    if "MAY2025" in path.name
)

if not PROCESSED_DATASETS:
    raise FileNotFoundError(f"No processed CSV files found in {PROCESSED_DATASET_DIR} containing 'MAY2025'.")

TRUE_PROCESSED_NAMES = [path.stem.rsplit("_", 1)[-1] for path in PROCESSED_DATASETS]

missing_processed = [tl for tl in TIMELAPSE_INFORMATION if tl not in TRUE_PROCESSED_NAMES]
if missing_processed:
    raise ValueError(
        "Timelapse IDs missing in processed datasets: "
        + ", ".join(missing_processed)
    )


In [ ]:
def export_rois_data(rois, dataset, dataset_name=None, width=1, offset=0):
    """
    Export ROI data (means, stds, schedules) from a fluopti dataset to CSV.
    """
    data_dict = {}

    if dataset_name is None:
        dataset_name = getattr(dataset, "name", "dataset")

    for roi_index, roi in enumerate(rois):
        tpoints = rois[roi].times["W"]

        time.sleep(1)

        experimental_data_means, experimental_data_std = flua.border_signal(
            rois[roi], wide=width, offset=offset
        )
        experimental_data_means = np.nan_to_num(experimental_data_means)
        experimental_data_std = np.nan_to_num(experimental_data_std)

        t0_tf = np.array([tpoints[0], tpoints[-1]], dtype=np.float64)
        data_tpoints = np.linspace(tpoints[0], tpoints[-1], len(tpoints))
        IA_0 = np.array([experimental_data_means[0]], dtype=np.float64)

        schedule_times = np.array(dataset.control_regime["T"])
        schedule_R = np.array([int(i / 100) for i in dataset.control_regime["R"]])
        schedule_G = np.array([int(i / 100) for i in dataset.control_regime["G"]])

        schedule_R_times = np.zeros(len(data_tpoints), dtype=np.float64)
        schedule_G_times = np.zeros(len(data_tpoints), dtype=np.float64)

        for i, timepoint in enumerate(data_tpoints):
            step = next((idx for idx, time_limit in enumerate(schedule_times) if timepoint < time_limit), len(schedule_times) - 1)
            schedule_R_times[i] = schedule_R[step]
            schedule_G_times[i] = schedule_G[step]

        roi_key = str(roi_index)
        data_dict[roi_key] = {
            "roi": int(roi_key),
            "time": tpoints,
            "exp_data_means": experimental_data_means,
            "exp_data_stds": experimental_data_std,
            "t0_tf": t0_tf,
            "data_tpoints": data_tpoints,
            "IA_0": IA_0,
            "schedule_times": schedule_times,
            "schedule_R": schedule_R,
            "schedule_G": schedule_G,
            "schedule_R_COMPLETE": schedule_R_times,
            "schedule_G_COMPLETE": schedule_G_times,
            "firstswitch": np.array([schedule_times[0]] * len(tpoints), dtype=np.float64),
            "secondswitch": np.array([schedule_times[1]] * len(tpoints), dtype=np.float64),
        }

    data_df = pd.DataFrame(data_dict)
    out_path = working_path / "data_analysis" / f"EXTRACTED_DATA_{dataset_name}.csv"
    data_df.to_csv(out_path, index=False)
    return data_df


In [ ]:
def parse_array_string(s: str) -> np.ndarray:
    """
    Convert a bracketed string of numbers into a 1-D float array.
    Handles whitespace-, comma-, or ellipsis-separated lists.
    Returns an empty array if no numbers are found.
    """
    core = s.strip()
    if core.startswith("[") and core.endswith("]"):
        core = core[1:-1]

    number_regex = r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?"
    nums = re.findall(number_regex, core)
    return np.asarray(nums, dtype=np.float64)


In [ ]:
ROW_NAMES = [
    "roi", "time", "exp_data_means", "exp_data_stds",
    "t0_tf", "data_tpoints", "IA_0",
    "schedule_times", "schedule_R", "schedule_G",
    "schedule_R_COMPLETE", "schedule_G_COMPLETE",
    "firstswitch", "secondswitch",
    "dataset_name",
]

def load_and_patch_dataset(csv_path: Path, green_pct: float, red_pct: float, max_rois: int = 50, max_len: int = 192, zero_plateau: int = 80):
    """
    Clean one MAY-2025 ROI table and standardize schedule vectors.
    """
    tmp = pd.read_csv(csv_path)

    expected_rows = len(ROW_NAMES) - 1
    if tmp.shape[0] != expected_rows:
        raise ValueError(f"Unexpected row count in {csv_path}: {tmp.shape[0]} (expected {expected_rows}).")

    tmp.index = ROW_NAMES[:-1]

    valid_cols = []
    for c in (col for col in tmp.columns if str(col).isdigit()):
        try:
            means_arr = parse_array_string(tmp.at["exp_data_means", c])
        except Exception as err:
            print(f"[skip] {csv_path} ROI {c}: parse error ({err})")
            continue

        if np.any(means_arr < 0):
            continue

        if means_arr.size == 0:
            continue

        if means_arr.size > zero_plateau and np.any(means_arr[zero_plateau:] == 0):
            continue

        valid_cols.append(c)

    if not valid_cols:
        print(f"[skip] {csv_path}: no ROI passed the QC filters.")
        return None

    sampled_cols = random.sample(valid_cols, min(max_rois, len(valid_cols)))
    tmp = tmp[sampled_cols]

    for c in sampled_cols:
        time_arr = parse_array_string(tmp.at["time", c])
        means_arr = parse_array_string(tmp.at["exp_data_means", c])
        stds_arr = parse_array_string(tmp.at["exp_data_stds", c])
        data_tpts = parse_array_string(tmp.at["data_tpoints", c])

        if time_arr.size > max_len:
            slice_ = slice(0, max_len)
            time_arr = time_arr[slice_]
            means_arr = means_arr[slice_]
            stds_arr = stds_arr[slice_]
            data_tpts = data_tpts[slice_]

            fs_arr = parse_array_string(tmp.at["firstswitch", c])[:max_len]
            ss_arr = parse_array_string(tmp.at["secondswitch", c])[:max_len]
            tmp.at["firstswitch", c] = str(list(fs_arr))
            tmp.at["secondswitch", c] = str(list(ss_arr))

            t0_tf = parse_array_string(tmp.at["t0_tf", c])
            t0_tf[1] = time_arr[-1]
            tmp.at["t0_tf", c] = str(list(t0_tf))

        tmp.at["time", c] = str(list(time_arr))
        tmp.at["exp_data_means", c] = str(list(means_arr))
        tmp.at["exp_data_stds", c] = str(list(stds_arr))
        tmp.at["data_tpoints", c] = str(list(data_tpts))

        n_tpts = time_arr.size
        red_vec = str(list(np.full(n_tpts, red_pct)))
        green_vec = str(list(np.full(n_tpts, green_pct)))

        tmp.at["schedule_R_COMPLETE", c] = red_vec
        tmp.at["schedule_G_COMPLETE", c] = green_vec
        tmp.at["schedule_R", c] = str([red_pct])
        tmp.at["schedule_G", c] = str([green_pct])

    dataset_key = csv_path.stem.split("_")[-1]
    tmp.loc["dataset_name"] = dataset_key

    return tmp


In [ ]:
all_tables = []

for csv_path in PROCESSED_DATASETS:
    dataset_key = csv_path.stem.rsplit("_", 1)[-1]

    if dataset_key not in TIMELAPSE_INFORMATION:
        raise KeyError(f"{dataset_key} not in TIMELAPSE_INFORMATION. Please add it or rename the file.")

    green_pct, red_pct = TIMELAPSE_INFORMATION[dataset_key][1]   # [G, R]
    patched = load_and_patch_dataset(csv_path, green_pct, red_pct, max_rois=75)
    if patched is None:
        continue

    patched.columns = [f"{dataset_key}_{c}" for c in patched.columns]
    all_tables.append(patched)

if not all_tables:
    raise RuntimeError("No ROI tables survived the QC filters.")

aggregated = pd.concat(all_tables, axis=1)
AGGREGATED_DATA_PATH = PROCESSED_DATASET_DIR / "AGGREGATED_ROI_DATA.csv"
aggregated.to_csv(AGGREGATED_DATA_PATH, index=True)
print(f"Aggregated ROI table written to {AGGREGATED_DATA_PATH}")


In [ ]:
print(f"Aggregated shape: {aggregated.shape}")
print(f"First five columns: {aggregated.columns[:5].tolist()}")


In [ ]:
# Plot every ROI (means +/- SD) capped at 48 h (index 192)
plt.figure(figsize=(10, 6))

for col in aggregated.columns:
    times = parse_array_string(aggregated.at["time", col])
    means = parse_array_string(aggregated.at["exp_data_means", col])
    stds = parse_array_string(aggregated.at["exp_data_stds", col])

    plt.plot(times, means, alpha=0.6)
    plt.fill_between(times, means - stds, means + stds, alpha=0.15)

plt.xlabel("Time (min)")
plt.ylabel("Fluorescence (a.u.)")
plt.title("Experimental mean +/- SD for each sampled ROI (capped at 48 h)")
plt.tight_layout()
plt.show()


In [ ]:
def lookup_light_schedule(exp_id: str, t: np.ndarray):
    """
    Return the green and red-light intensity vectors (same length as `t`)
    for the requested experiment ID.
    """
    try:
        green_frac, red_frac = TIMELAPSE_INFORMATION[exp_id][1]
    except KeyError:
        raise ValueError(f"Unknown experiment ID '{exp_id}'. Please add it to TIMELAPSE_INFORMATION.")

    I_G = np.full_like(t, green_frac, dtype=np.float64)
    I_R = np.full_like(t, red_frac, dtype=np.float64)
    return I_G, I_R


In [ ]:
if not AGGREGATED_DATA_PATH.exists():
    raise FileNotFoundError(f"{AGGREGATED_DATA_PATH} not found. Run the aggregation cell first.")

df = pd.read_csv(AGGREGATED_DATA_PATH, index_col=0)

records = []

for col in df.columns:
    exp_id, roi_id = col.split("_")
    roi_id = int(roi_id)

    t_vec = parse_array_string(df.at["time", col])
    y_vec = parse_array_string(df.at["exp_data_means", col])

    first_valid = 0
    t_vec = t_vec[first_valid:]
    y_vec = y_vec[first_valid:]

    if len(t_vec) != len(np.unique(t_vec)):
        uniq_mask = np.concatenate(([True], np.diff(t_vec) != 0))
        t_vec = t_vec[uniq_mask]
        y_vec = y_vec[uniq_mask]

    I_G_vec, I_R_vec = lookup_light_schedule(exp_id, t_vec)
    sigma = max(1.0, np.std(y_vec[:10])) if y_vec.size else 1.0

    records.append(dict(
        exp=exp_id,
        roi=roi_id,
        t=t_vec,
        y=y_vec,
        I_G=I_G_vec,
        I_R=I_R_vec,
        u0=np.array([y_vec[0]], np.float64) if y_vec.size else np.array([0.0], np.float64),
        sigma=sigma,
    ))

big_dataset = records
print(f"Loaded {len(big_dataset)} colony traces")


In [ ]:
@nb.njit(inline="always")
def _bin_search(t_arr, t_val):
    left = 0
    right = t_arr.size - 1
    while left < right:
        mid = (left + right) // 2
        if t_arr[mid] < t_val:
            left = mid + 1
        else:
            right = mid
    return left

@nb.njit(inline="always")
def light_interp_arr(t_arr, g_arr, r_arr, t_val):
    idx = _bin_search(t_arr, t_val)
    if idx >= t_arr.size:
        idx = t_arr.size - 1
    return g_arr[idx], r_arr[idx]

_lt = np.empty(1, dtype=np.float64)
_lg = np.empty_like(_lt)
_lr = np.empty_like(_lt)


In [ ]:
sample_indices = [i for i in (0, 50, 100, 150, 200) if i < len(big_dataset)]

for idx in sample_indices:
    rec = big_dataset[idx]
    t_vec = rec["t"]
    g_vec = rec["I_G"]
    r_vec = rec["I_R"]

    print(f"\nEXP {rec['exp']}  ROI {rec['roi']}  len={t_vec.size}")
    print("first light :", light_interp_arr(t_vec, g_vec, r_vec, t_vec[0]))
    print("last  light :", light_interp_arr(t_vec, g_vec, r_vec, t_vec[-1]))
